# Enriquecimento do Dataset com Dados Externos

Este notebook tem como objetivo enriquecer o dataset analítico do projeto com informações externas provenientes de fontes públicas e oficiais.

O enriquecimento busca incorporar variáveis territoriais, populacionais, econômicas e educacionais que possam contribuir para a compreensão dos fatores associados à alfabetização.

Nesta primeira etapa serão utilizados dados municipais do IBGE, posteriormente integrados ao dataset analítico por meio do código oficial do município.

As principais variáveis consideradas são:

- região geográfica;
- população;
- área territorial;
- densidade demográfica;
- PIB per capita.

O dataset enriquecido será utilizado posteriormente nas etapas de Análise Exploratória de Dados e Machine Learning.

## 1. Importação das bibliotecas

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import requests

## 2. Definição dos caminhos

O dataset analítico construído anteriormente será utilizado como base para o enriquecimento.

In [2]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data"
GOLD_DIR = DATA_DIR / "gold"
EXTERNAL_DIR = DATA_DIR / "external"

EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = GOLD_DIR / "gold_dataset_analitico.parquet"

df = pd.read_parquet(DATASET_PATH)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

display(df.head())

Linhas: 57,782
Colunas: 17


,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,sigla_uf,meta_municipio_ano,meta_uf_ano,percentual_participacao
0,2023,1302504,Manacapuru,60000951,13015851,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,AM,NaN,NaN,87.06
1,2023,1302603,Manaus,60000963,13030738,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,AM,NaN,NaN,71.33
2,2023,1300631,Beruri,60001351,13003982,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,AM,NaN,NaN,NaN
3,2023,1711506,Jaú do Tocantins,60004115,17012510,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,TO,NaN,NaN,82.98
4,2023,2100709,Anajatuba,60004434,21012344,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,Não,NaN,NaN,MA,NaN,NaN,91.54


## 3. Enriquecimento territorial

A região geográfica é derivada da Unidade Federativa presente no dataset.

Essa informação permite realizar análises territoriais em nível de macrorregião e pode ser utilizada como variável categórica nos modelos posteriores.

In [3]:
mapa_regiao = {
    "AC": "Norte",
    "AP": "Norte",
    "AM": "Norte",
    "PA": "Norte",
    "RO": "Norte",
    "RR": "Norte",
    "TO": "Norte",

    "AL": "Nordeste",
    "BA": "Nordeste",
    "CE": "Nordeste",
    "MA": "Nordeste",
    "PB": "Nordeste",
    "PE": "Nordeste",
    "PI": "Nordeste",
    "RN": "Nordeste",
    "SE": "Nordeste",

    "DF": "Centro-Oeste",
    "GO": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "MS": "Centro-Oeste",

    "ES": "Sudeste",
    "MG": "Sudeste",
    "RJ": "Sudeste",
    "SP": "Sudeste",

    "PR": "Sul",
    "RS": "Sul",
    "SC": "Sul"
}

df["regiao"] = df["sigla_uf"].map(mapa_regiao)

In [4]:
display(
    df["regiao"]
    .value_counts(dropna=False)
)

regiao
Sudeste         18050
Nordeste        16208
Sul             10927
Norte            6693
Centro-Oeste     5836
NaN                68
Name: count, dtype: int64

## 4. Contexto econômico municipal

O PIB per capita municipal será utilizado como uma variável de contexto econômico.

Para reduzir o risco de utilização de informações futuras, será adotado o PIB per capita do ano imediatamente anterior ao ano da avaliação.

Dessa forma:

- registros de 2023 utilizam o PIB per capita de 2022;
- registros de 2024 utilizam o PIB per capita de 2023.

Essa estratégia mantém maior coerência temporal para a etapa de modelagem.

### 4.1 Download da base de PIB dos Municípios

Os dados de PIB municipal serão obtidos diretamente da base oficial do IBGE.

Será utilizada a base consolidada de 2010 a 2023, que contém informações econômicas em nível municipal.

Para preservar coerência temporal na modelagem, será utilizado o PIB per capita do ano imediatamente anterior ao ano da avaliação.

In [5]:
import io
import zipfile
import requests
import pandas as pd

URL_PIB = (
    "https://ftp.ibge.gov.br/Pib_Municipios/"
    "2022_2023/base/base_de_dados_2010_2023_xlsx.zip"
)

response = requests.get(URL_PIB, timeout=120)
response.raise_for_status()

print(f"Download concluído: {len(response.content) / 1024**2:.2f} MB")

Download concluído: 19.74 MB


In [6]:
with zipfile.ZipFile(io.BytesIO(response.content)) as arquivo_zip:
    print("Arquivos encontrados:")
    
    for nome in arquivo_zip.namelist():
        print("-", nome)

    arquivo_zip.extractall(EXTERNAL_DIR)

Arquivos encontrados:
- PIB dos Municípios - base de dados 2010-2023.xlsx


In [7]:
arquivos_extraidos = list(EXTERNAL_DIR.glob("*.xlsx"))

for arquivo in arquivos_extraidos:
    print(arquivo.name)

PIB dos Municípios - base de dados 2010-2023.xlsx


In [8]:
arquivo_pib = arquivos_extraidos[0]

xls = pd.ExcelFile(arquivo_pib)

print("Planilhas disponíveis:")
print(xls.sheet_names)

Planilhas disponíveis:
['PIB dos Municípios', 'Notas']


In [9]:
df_pib_raw = pd.read_excel(
    arquivo_pib,
    sheet_name=0
)

print("Shape:", df_pib_raw.shape)

display(df_pib_raw.head())

print("\nColunas:")
print(df_pib_raw.columns.tolist())

Shape: (77965, 43)


,Ano,Código da Grande Região,Nome da Grande Região,Código da Unidade da Federação,Sigla da Unidade da Federação,Nome da Unidade da Federação,Código do Município,Nome do Município,Região Metropolitana,Código da Mesorregião,...,"Valor adicionado bruto da Indústria,\na preços correntes\n(R$ 1.000)","Valor adicionado bruto dos Serviços,\na preços correntes \n- exceto Administração, defesa, educação e saúde públicas e seguridade social\n(R$ 1.000)","Valor adicionado bruto da Administração, defesa, educação e saúde públicas e seguridade social, \na preços correntes\n(R$ 1.000)","Valor adicionado bruto total, \na preços correntes\n(R$ 1.000)","Impostos, líquidos de subsídios, sobre produtos, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)",Atividade com maior valor adicionado bruto,Atividade com segundo maior valor adicionado bruto,Atividade com terceiro maior valor adicionado bruto
0,2010,1,Norte,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,NaN,1102,...,16118.534,62496.185,93244.656,241119.767,20957.111,262076.878,10731.18,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
1,2010,1,Norte,11,RO,Rondônia,1100023,Ariquemes,NaN,1102,...,287138.585,494946.267,343867.731,1199664.227,165029.553,1364693.780,15103.86,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
2,2010,1,Norte,11,RO,Rondônia,1100031,Cabixi,NaN,1102,...,3252.506,12677.210,25170.235,65400.772,4210.342,69611.114,11033.62,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
3,2010,1,Norte,11,RO,Rondônia,1100049,Cacoal,NaN,1102,...,182051.537,465447.325,298454.309,1041212.374,145281.717,1186494.091,15095.15,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
4,2010,1,Norte,11,RO,Rondônia,1100056,Cerejeiras,NaN,1102,...,19734.484,80724.991,63018.270,192454.160,29567.029,222021.189,13037.06,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...



Colunas:
['Ano', 'Código da Grande Região', 'Nome da Grande Região', 'Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação', 'Código do Município', 'Nome do Município', 'Região Metropolitana', 'Código da Mesorregião', 'Nome da Mesorregião', 'Código da Microrregião', 'Nome da Microrregião', 'Código da Região Geográfica Imediata', 'Nome da Região Geográfica Imediata', 'Município da Região Geográfica Imediata', 'Código da Região Geográfica Intermediária', 'Nome da Região Geográfica Intermediária', 'Município da Região Geográfica Intermediária', 'Código Concentração Urbana', 'Nome Concentração Urbana', 'Tipo Concentração Urbana', 'Código Arranjo Populacional', 'Nome Arranjo Populacional', 'Hierarquia Urbana', 'Hierarquia Urbana (principais categorias)', 'Código da Região Rural', 'Nome da Região Rural', 'Região rural (segundo classificação do núcleo)', 'Amazônia Legal', 'Semiárido', 'Cidade-Região de São Paulo', 'Valor adicionado bruto da Agropecu

### 4.4 Seleção das variáveis econômicas

A base do PIB dos Municípios contém diversas informações territoriais e econômicas.

Para este projeto, serão mantidas apenas as variáveis necessárias para o enriquecimento do dataset analítico:

- ano de referência;
- código do município;
- PIB per capita.

Essa seleção reduz a complexidade da base e mantém apenas informações relevantes para a análise e modelagem.

In [10]:
df_pib = df_pib_raw[
    [
        "Ano",
        "Código do Município",
        "Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)"
    ]
].copy()

In [11]:
display(df_pib.head())

print(df_pib.dtypes)

,Ano,Código do Município,"Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)"
0,2010,1100015,10731.18
1,2010,1100023,15103.86
2,2010,1100031,11033.62
3,2010,1100049,15095.15
4,2010,1100056,13037.06


Ano                                                                    int64
Código do Município                                                    int64
Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)    float64
dtype: object


In [12]:
df_pib = df_pib.rename(
    columns={
        "Ano": "ano_pib",
        "Código do Município": "id_municipio",
        "Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)": "pib_per_capita"
    }
)

print(df_pib.columns.tolist())
display(df_pib.head())

['ano_pib', 'id_municipio', 'pib_per_capita']


,ano_pib,id_municipio,pib_per_capita
0,2010,1100015,10731.18
1,2010,1100023,15103.86
2,2010,1100031,11033.62
3,2010,1100049,15095.15
4,2010,1100056,13037.06


### 4.5 Padronização dos tipos

Para garantir compatibilidade com o dataset analítico principal, o código do município será tratado como identificador textual e as variáveis numéricas serão convertidas para tipos apropriados.

In [13]:
df_pib["id_municipio"] = (
    df_pib["id_municipio"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(7)
)

df_pib["ano_pib"] = pd.to_numeric(
    df_pib["ano_pib"],
    errors="coerce"
).astype("Int64")

df_pib["pib_per_capita"] = pd.to_numeric(
    df_pib["pib_per_capita"],
    errors="coerce"
)

In [14]:
print(df_pib.dtypes)

display(df_pib.head())

ano_pib             Int64
id_municipio       string
pib_per_capita    float64
dtype: object


,ano_pib,id_municipio,pib_per_capita
0,2010,1100015,10731.18
1,2010,1100023,15103.86
2,2010,1100031,11033.62
3,2010,1100049,15095.15
4,2010,1100056,13037.06


### 4.6 Seleção dos anos de interesse

Como o dataset analítico contém avaliações de 2023 e 2024, serão utilizados os valores de PIB per capita do ano imediatamente anterior a cada avaliação.

Dessa forma:

- avaliações de 2023 utilizam o PIB per capita de 2022;
- avaliações de 2024 utilizam o PIB per capita de 2023.

Essa estratégia busca manter coerência temporal e reduzir o risco de utilização de informações futuras na modelagem.

In [15]:
df_pib = df_pib[
    df_pib["ano_pib"].isin([2022, 2023])
].copy()

In [16]:
display(
    df_pib["ano_pib"]
    .value_counts()
    .sort_index()
)

ano_pib
2022    5570
2023    5570
Name: count, dtype: Int64

In [17]:
print("Registros:", len(df_pib))
print("Municípios únicos:", df_pib["id_municipio"].nunique())

Registros: 11140
Municípios únicos: 5570


### 4.7 Associação temporal com o ano da avaliação

Para garantir coerência temporal, o PIB per capita será associado ao ano seguinte da avaliação.

Assim:

- PIB de 2022 será utilizado nos registros de avaliação de 2023;
- PIB de 2023 será utilizado nos registros de avaliação de 2024.

Essa transformação permite integrar a informação econômica utilizando apenas dados anteriores ao período avaliado.

In [18]:
df_pib["ano"] = df_pib["ano_pib"] + 1

In [19]:
display(
    df_pib[
        [
            "id_municipio",
            "ano_pib",
            "ano",
            "pib_per_capita"
        ]
    ].head()
)

,id_municipio,ano_pib,ano,pib_per_capita
66825,1100015,2022,2023,42778.33
66826,1100023,2022,2023,39339.43
66827,1100031,2022,2023,54033.69
66828,1100049,2022,2023,36774.15
66829,1100056,2022,2023,56834.39


In [20]:
print(
    "Duplicidades por município + ano:",
    df_pib.duplicated(
        subset=["id_municipio", "ano"]
    ).sum()
)

Duplicidades por município + ano: 0


### 4.8 Integração do PIB per capita

Após o alinhamento temporal, o PIB per capita é integrado ao dataset analítico utilizando as chaves:

- `id_municipio`;
- `ano`.

A integração utiliza uma relação `many-to-one`, pois vários alunos podem pertencer ao mesmo município e ano, enquanto a base econômica possui apenas um registro por município em cada período.

In [21]:
df_pib_contexto = df_pib[
    [
        "id_municipio",
        "ano",
        "pib_per_capita"
    ]
].copy()

In [22]:
df_enriquecido = df.merge(
    df_pib_contexto,
    on=["id_municipio", "ano"],
    how="left",
    validate="many_to_one"
)

In [23]:
print("Antes do join:", df.shape)
print("Depois do join:", df_enriquecido.shape)

print(
    "Diferença de registros:",
    len(df_enriquecido) - len(df)
)

Antes do join: (57782, 18)
Depois do join: (57782, 19)
Diferença de registros: 0


In [24]:
print(
    "PIB per capita preenchido:",
    round(
        df_enriquecido["pib_per_capita"]
        .notna()
        .mean() * 100,
        2
    ),
    "%"
)

PIB per capita preenchido: 100.0 %


In [25]:
cobertura_pib = (
    df_enriquecido
    .groupby("ano")
    .agg(
        alunos=("id_aluno", "size"),
        pct_pib_preenchido=(
            "pib_per_capita",
            lambda x: x.notna().mean() * 100
        )
    )
    .round(2)
)

display(cobertura_pib)

,alunos,pct_pib_preenchido
ano,,
2023,28295,100.0
2024,29487,100.0


### 4.9 Validação da integração econômica

A integração do PIB per capita foi concluída com sucesso.

O join preservou integralmente a quantidade de registros da base individual e apresentou cobertura de 100% nos anos de 2023 e 2024.

Dessa forma, a variável `pib_per_capita` poderá ser utilizada posteriormente nas análises exploratórias e na avaliação de sua relevância para os modelos preditivos.

## 5. Contexto populacional e territorial

Além do contexto econômico, serão incorporadas variáveis estruturais dos municípios provenientes do IBGE.

Nesta etapa serão consideradas:

- população residente;
- área territorial;
- densidade demográfica.

Essas informações ajudam a representar características demográficas e territoriais dos municípios e serão utilizadas posteriormente na Análise Exploratória de Dados e na avaliação das features para Machine Learning.

Como essas variáveis representam características estruturais do município, serão utilizados dados do Censo Demográfico de 2022.

### 5.1 População residente

A população residente será obtida diretamente do IBGE em nível municipal.

O código oficial do município será preservado como chave de integração com o dataset analítico.

In [26]:
import requests

URL_POPULACAO = (
    "https://apisidra.ibge.gov.br/values/"
    "t/4714/n6/all/v/93/p/2022"
)

response = requests.get(
    URL_POPULACAO,
    timeout=120
)

print("Status:", response.status_code)
print("Tamanho da resposta:", len(response.content))

Status: 200
Tamanho da resposta: 1447059


### 5.2 Transformação da resposta em DataFrame

Após a confirmação de acesso à API do SIDRA/IBGE, os dados retornados serão convertidos para um DataFrame.

Nesta etapa, o objetivo é inspecionar a estrutura da resposta e identificar as colunas correspondentes ao código do município e à população residente.

In [27]:
dados_populacao = response.json()

df_pop_raw = pd.DataFrame(dados_populacao)

print("Shape:", df_pop_raw.shape)
print("\nColunas:")
print(df_pop_raw.columns.tolist())

display(df_pop_raw.head())

Shape: (5571, 11)

Colunas:
['NC', 'NN', 'MC', 'MN', 'V', 'D1C', 'D1N', 'D2C', 'D2N', 'D3C', 'D3N']


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
1,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022
2,6,Município,45,Pessoas,96833,1100023,Ariquemes - RO,93,População residente,2022,2022
3,6,Município,45,Pessoas,5351,1100031,Cabixi - RO,93,População residente,2022,2022
4,6,Município,45,Pessoas,86887,1100049,Cacoal - RO,93,População residente,2022,2022


### 5.3 Preparação da população municipal

A resposta da API do SIDRA contém uma primeira linha descritiva com os significados das colunas.

Após sua remoção, serão mantidos apenas o código do município, o nome do município e a população residente em 2022.

O código oficial do município será padronizado para o mesmo formato utilizado no dataset analítico.

In [28]:
df_populacao = (
    df_pop_raw
    .iloc[1:]
    [["D1C", "D1N", "V"]]
    .copy()
)

df_populacao = df_populacao.rename(
    columns={
        "D1C": "id_municipio",
        "D1N": "municipio_ibge",
        "V": "populacao_2022"
    }
)

In [29]:
df_populacao["id_municipio"] = (
    df_populacao["id_municipio"]
    .astype("string")
    .str.zfill(7)
)

df_populacao["populacao_2022"] = pd.to_numeric(
    df_populacao["populacao_2022"],
    errors="coerce"
)

In [30]:
print("Shape:", df_populacao.shape)
print("Municípios únicos:", df_populacao["id_municipio"].nunique())
print("Nulos em população:", df_populacao["populacao_2022"].isna().sum())
print("Duplicidades:", df_populacao["id_municipio"].duplicated().sum())

display(df_populacao.head())

Shape: (5570, 3)
Municípios únicos: 5570
Nulos em população: 0
Duplicidades: 0


,id_municipio,municipio_ibge,populacao_2022
1,1100015,Alta Floresta D'Oeste - RO,21494
2,1100023,Ariquemes - RO,96833
3,1100031,Cabixi - RO,5351
4,1100049,Cacoal - RO,86887
5,1100056,Cerejeiras - RO,15890


### 5.4 Integração da população municipal

Após a preparação da base populacional do IBGE, a população residente de 2022 será integrada ao dataset analítico utilizando `id_municipio` como chave.

Como a população utilizada é uma característica estrutural do município e possui referência anterior aos anos de avaliação, ela será mantida como variável de contexto para EDA e modelagem.

In [31]:
df_enriquecido = df_enriquecido.merge(
    df_populacao[
        [
            "id_municipio",
            "populacao_2022"
        ]
    ],
    on="id_municipio",
    how="left",
    validate="many_to_one"
)

In [32]:
print("Shape após população:", df_enriquecido.shape)

print(
    "Cobertura população:",
    round(
        df_enriquecido["populacao_2022"]
        .notna()
        .mean() * 100,
        2
    ),
    "%"
)

print(
    "Nulos em população:",
    df_enriquecido["populacao_2022"].isna().sum()
)

Shape após população: (57782, 20)
Cobertura população: 100.0 %
Nulos em população: 0


### 5.5 Área territorial e densidade demográfica

Para complementar o contexto territorial dos municípios, serão avaliadas as variáveis de área territorial e densidade demográfica disponibilizadas pelo IBGE.

Antes da integração, será consultada a estrutura completa da tabela utilizada anteriormente para população, verificando quais variáveis estão disponíveis para o nível municipal.

In [33]:
URL_CENSO_COMPLETO = (
    "https://apisidra.ibge.gov.br/values/"
    "t/4714/n6/all/v/all/p/2022"
)

response_censo = requests.get(
    URL_CENSO_COMPLETO,
    timeout=120
)

print("Status:", response_censo.status_code)
print("Tamanho da resposta:", len(response_censo.content))

Status: 200
Tamanho da resposta: 4651939


In [34]:
dados_censo = response_censo.json()

df_censo_raw = pd.DataFrame(dados_censo)

print("Shape:", df_censo_raw.shape)

display(df_censo_raw.head(10))

Shape: (16711, 11)


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano
1,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022
2,6,Município,26,Quilômetros quadrados,7067.127,1100015,Alta Floresta D'Oeste - RO,6318,Área da unidade territorial,2022,2022
3,6,Município,28,Habitante por quilômetro quadrado,3.04,1100015,Alta Floresta D'Oeste - RO,614,Densidade demográfica,2022,2022
4,6,Município,45,Pessoas,96833,1100023,Ariquemes - RO,93,População residente,2022,2022
5,6,Município,26,Quilômetros quadrados,4426.571,1100023,Ariquemes - RO,6318,Área da unidade territorial,2022,2022
6,6,Município,28,Habitante por quilômetro quadrado,21.88,1100023,Ariquemes - RO,614,Densidade demográfica,2022,2022
7,6,Município,45,Pessoas,5351,1100031,Cabixi - RO,93,População residente,2022,2022
8,6,Município,26,Quilômetros quadrados,1314.352,1100031,Cabixi - RO,6318,Área da unidade territorial,2022,2022
9,6,Município,28,Habitante por quilômetro quadrado,4.07,1100031,Cabixi - RO,614,Densidade demográfica,2022,2022


In [35]:
display(
    df_censo_raw[
        ["D2C", "D2N"]
    ]
    .drop_duplicates()
)

,D2C,D2N
0,Variável (Código),Variável
1,93,População residente
2,6318,Área da unidade territorial
3,614,Densidade demográfica


### 5.6 Extração das variáveis territoriais

A tabela do SIDRA disponibiliza, em nível municipal, três variáveis relevantes para o contexto territorial:

- população residente;
- área da unidade territorial;
- densidade demográfica.

Essas variáveis serão extraídas e reorganizadas para formar uma única tabela municipal, posteriormente integrada ao dataset analítico.

In [36]:
df_censo = (
    df_censo_raw
    .iloc[1:]
    [["D1C", "D1N", "D2C", "D2N", "V"]]
    .copy()
)

df_censo = df_censo.rename(
    columns={
        "D1C": "id_municipio",
        "D1N": "municipio_ibge",
        "D2C": "codigo_variavel",
        "D2N": "variavel",
        "V": "valor"
    }
)

In [37]:
df_censo["id_municipio"] = (
    df_censo["id_municipio"]
    .astype("string")
    .str.zfill(7)
)

df_censo["codigo_variavel"] = (
    df_censo["codigo_variavel"]
    .astype("string")
)

df_censo["valor"] = pd.to_numeric(
    df_censo["valor"],
    errors="coerce"
)

In [38]:
display(df_censo.head(10))

print(df_censo.dtypes)

,id_municipio,municipio_ibge,codigo_variavel,variavel,valor
1,1100015,Alta Floresta D'Oeste - RO,93,População residente,21494.000
2,1100015,Alta Floresta D'Oeste - RO,6318,Área da unidade territorial,7067.127
3,1100015,Alta Floresta D'Oeste - RO,614,Densidade demográfica,3.040
4,1100023,Ariquemes - RO,93,População residente,96833.000
5,1100023,Ariquemes - RO,6318,Área da unidade territorial,4426.571
6,1100023,Ariquemes - RO,614,Densidade demográfica,21.880
7,1100031,Cabixi - RO,93,População residente,5351.000
8,1100031,Cabixi - RO,6318,Área da unidade territorial,1314.352
9,1100031,Cabixi - RO,614,Densidade demográfica,4.070
10,1100049,Cacoal - RO,93,População residente,86887.000


id_municipio        string
municipio_ibge         str
codigo_variavel     string
variavel               str
valor              float64
dtype: object


### 5.7 Reorganização das variáveis territoriais

Os dados retornados pelo SIDRA estão em formato longo, com uma linha para cada variável de cada município.

Para facilitar a integração com o dataset analítico, as variáveis serão reorganizadas em colunas distintas:

- `populacao_2022`;
- `area_km2`;
- `densidade_demografica`.

In [39]:
df_territorial = (
    df_censo
    .pivot_table(
        index=["id_municipio", "municipio_ibge"],
        columns="codigo_variavel",
        values="valor",
        aggfunc="first"
    )
    .reset_index()
)

In [40]:
df_territorial = df_territorial.rename(
    columns={
        "93": "populacao_2022",
        "6318": "area_km2",
        "614": "densidade_demografica"
    }
)

In [41]:
display(df_territorial.head())

print("Shape:", df_territorial.shape)
print("Municípios únicos:", df_territorial["id_municipio"].nunique())

print("\nNulos:")
display(
    df_territorial[
        [
            "populacao_2022",
            "area_km2",
            "densidade_demografica"
        ]
    ].isna().sum()
)

codigo_variavel,id_municipio,municipio_ibge,densidade_demografica,area_km2,populacao_2022
0,1100015,Alta Floresta D'Oeste - RO,3.04,7067.127,21494.0
1,1100023,Ariquemes - RO,21.88,4426.571,96833.0
2,1100031,Cabixi - RO,4.07,1314.352,5351.0
3,1100049,Cacoal - RO,22.91,3793.000,86887.0
4,1100056,Cerejeiras - RO,5.71,2783.300,15890.0


Shape: (5570, 5)
Municípios únicos: 5570

Nulos:


codigo_variavel
populacao_2022           0
area_km2                 0
densidade_demografica    0
dtype: int64

### 5.8 Integração das variáveis territoriais

Após a reorganização dos dados do Censo 2022, as variáveis de população, área territorial e densidade demográfica serão integradas ao dataset analítico por meio do código oficial do município.

A integração utiliza uma relação `many-to-one`, pois vários alunos podem pertencer ao mesmo município, enquanto a base territorial possui apenas um registro por município.

In [42]:
df_territorial_contexto = df_territorial[
    [
        "id_municipio",
        "area_km2",
        "densidade_demografica"
    ]
].copy()

In [43]:
df_enriquecido = df_enriquecido.merge(
    df_territorial_contexto,
    on="id_municipio",
    how="left",
    validate="many_to_one"
)

In [44]:
print("Shape após contexto territorial:", df_enriquecido.shape)

for coluna in ["area_km2", "densidade_demografica"]:
    print(
        f"{coluna} - cobertura:",
        round(
            df_enriquecido[coluna].notna().mean() * 100,
            2
        ),
        "%"
    )

Shape após contexto territorial: (57782, 22)
area_km2 - cobertura: 100.0 %
densidade_demografica - cobertura: 100.0 %


### 5.9 Validação do enriquecimento territorial

A integração das variáveis territoriais foi concluída com sucesso.

As informações de área territorial e densidade demográfica apresentaram cobertura de 100% na base individual, sem alteração na quantidade de registros.

Com isso, o dataset passa a incorporar variáveis econômicas, populacionais e territoriais relevantes para as etapas posteriores de análise e modelagem.

## 6. Contexto escolar e infraestrutura educacional

Após o enriquecimento econômico, populacional e territorial, o dataset será complementado com informações provenientes do Censo Escolar da Educação Básica.

Os microdados do Censo Escolar, disponibilizados pelo INEP, permitem incorporar características relacionadas à infraestrutura, localização e organização das escolas.

Serão utilizados os arquivos referentes aos anos de 2023 e 2024, compatíveis com o período presente no dataset analítico.

A integração será realizada posteriormente por meio do identificador da escola e do ano de referência.

### 6.1 Definição dos arquivos locais

Os microdados oficiais do Censo Escolar de 2023 e 2024 foram obtidos no portal do INEP e armazenados localmente na pasta `data/external`.

Os arquivos permanecerão compactados inicialmente, permitindo a inspeção do conteúdo antes da leitura dos dados.

In [45]:
from pathlib import Path
import zipfile
import pandas as pd

CENSO_2023_PATH = (
    EXTERNAL_DIR / "microdados_censo_escolar_2023.zip"
)

CENSO_2024_PATH = (
    EXTERNAL_DIR / "microdados_censo_escolar_2024.zip"
)

In [46]:
for caminho in [CENSO_2023_PATH, CENSO_2024_PATH]:
    print(
        caminho.name,
        "| Existe:",
        caminho.exists(),
        "| Tamanho:",
        round(caminho.stat().st_size / 1024**2, 2)
        if caminho.exists()
        else "Arquivo não encontrado"
    )

microdados_censo_escolar_2023.zip | Existe: True | Tamanho: 30.61
microdados_censo_escolar_2024.zip | Existe: True | Tamanho: 32.26


### 6.2 Inspeção do conteúdo dos arquivos

Antes de carregar os microdados, será inspecionada a estrutura interna dos arquivos compactados.

Essa etapa permite identificar exatamente quais arquivos contêm os dados e a documentação necessária, evitando a extração desnecessária de todo o conteúdo.

In [47]:
with zipfile.ZipFile(CENSO_2024_PATH, "r") as zip_2024:
    arquivos_2024 = zip_2024.namelist()

print("Arquivos encontrados no Censo Escolar 2024:\n")

for arquivo in arquivos_2024:
    print(arquivo)

Arquivos encontrados no Censo Escolar 2024:

microdados_censo_escolar_2024_defeso/
microdados_censo_escolar_2024_defeso/Anexos/
microdados_censo_escolar_2024_defeso/Anexos/ANEXO I - Dicionário de Dados/
microdados_censo_escolar_2024_defeso/Anexos/ANEXO I - Dicionário de Dados/dicionário_dados_educação_básica.xlsx
microdados_censo_escolar_2024_defeso/Anexos/ANEXO I - Dicionário de Dados/~$Dicionário de Dados da Educação Básica.xlsx
microdados_censo_escolar_2024_defeso/Anexos/ANEXO I - Dicionário de Dados/~$dicionário_dados_educação_básica.xlsx
microdados_censo_escolar_2024_defeso/Anexos/ANEXO II -  Questionários do Censo da Educação Basica/
microdados_censo_escolar_2024_defeso/Anexos/ANEXO II -  Questionários do Censo da Educação Basica/Aluno 2024.pdf
microdados_censo_escolar_2024_defeso/Anexos/ANEXO II -  Questionários do Censo da Educação Basica/Escola 2024.pdf
microdados_censo_escolar_2024_defeso/Anexos/ANEXO II -  Questionários do Censo da Educação Basica/Escola Nova 2024.pdf
microd

In [48]:
with zipfile.ZipFile(CENSO_2023_PATH, "r") as zip_2023:
    arquivos_2023 = zip_2023.namelist()

print("Arquivos encontrados no Censo Escolar 2023:\n")

for arquivo in arquivos_2023:
    print(arquivo)

Arquivos encontrados no Censo Escolar 2023:

microdados_censo_escolar_2023/
microdados_censo_escolar_2023/Anexos/
microdados_censo_escolar_2023/Anexos/ANEXO I - Dicionário de Dados/
microdados_censo_escolar_2023/Anexos/ANEXO I - Dicionário de Dados/dicionário_dados_educaç╞o_básica.xlsx
microdados_censo_escolar_2023/Anexos/ANEXO II -  Questionários do Censo da Educaç╞o Basica/
microdados_censo_escolar_2023/Anexos/ANEXO II -  Questionários do Censo da Educaç╞o Basica/Aluno.pdf
microdados_censo_escolar_2023/Anexos/ANEXO II -  Questionários do Censo da Educaç╞o Basica/Escola.pdf
microdados_censo_escolar_2023/Anexos/ANEXO II -  Questionários do Censo da Educaç╞o Basica/Gestor Escolar.pdf
microdados_censo_escolar_2023/Anexos/ANEXO II -  Questionários do Censo da Educaç╞o Basica/Profissional Escolar.pdf
microdados_censo_escolar_2023/Anexos/ANEXO II -  Questionários do Censo da Educaç╞o Basica/Turma.pdf
microdados_censo_escolar_2023/dados/
microdados_censo_escolar_2023/dados/md5_microdados_ed_

In [49]:
csvs_2024 = [
    arquivo
    for arquivo in arquivos_2024
    if arquivo.lower().endswith(".csv")
]

csvs_2023 = [
    arquivo
    for arquivo in arquivos_2023
    if arquivo.lower().endswith(".csv")
]

print("CSV 2024:")
for arquivo in csvs_2024:
    print("-", arquivo)

print("\nCSV 2023:")
for arquivo in csvs_2023:
    print("-", arquivo)

CSV 2024:
- microdados_censo_escolar_2024_defeso/dados/microdados_ed_basica_2024.csv
- microdados_censo_escolar_2024_defeso/dados/suplemento_cursos_tecnicos_2024.csv

CSV 2023:
- microdados_censo_escolar_2023/dados/microdados_ed_basica_2023.csv
- microdados_censo_escolar_2023/dados/suplemento_cursos_tecnicos_2023.csv


### 6.5 Leitura inicial dos microdados

A partir da inspeção dos arquivos compactados, foi identificado o arquivo principal `microdados_ed_basica`, que contém as informações das escolas.

Inicialmente, será carregada apenas uma pequena amostra do arquivo de 2024 para identificar sua estrutura, tipos de dados e variáveis disponíveis.

In [50]:
ARQUIVO_CENSO_2024 = (
    "microdados_censo_escolar_2024_defeso/"
    "dados/microdados_ed_basica_2024.csv"
)

with zipfile.ZipFile(CENSO_2024_PATH, "r") as zip_2024:
    with zip_2024.open(ARQUIVO_CENSO_2024) as arquivo:
        df_censo_2024_amostra = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            nrows=5,
            low_memory=False
        )

print("Shape da amostra:", df_censo_2024_amostra.shape)

Shape da amostra: (5, 426)


In [51]:
df_censo_2024_amostra.head()

,NU_ANO_CENSO,NO_REGIAO,CO_REGIAO,NO_UF,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,NO_REGIAO_GEOG_INTERM,CO_REGIAO_GEOG_INTERM,...,QT_TUR_BAS_D,QT_TUR_BAS_N,QT_TUR_BAS_EAD,QT_TUR_INF_INT,QT_TUR_INF_CRE_INT,QT_TUR_INF_PRE_INT,QT_TUR_FUND_INT,QT_TUR_FUND_AI_INT,QT_TUR_FUND_AF_INT,QT_TUR_MED_INT
0,2024,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2024,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2024,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2024,Norte,1,Rondônia,RO,11,Alta Floresta D'Oeste,1100015,Ji-Paraná,1102,...,35.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [52]:
for coluna in df_censo_2024_amostra.columns:
    print(coluna)

NU_ANO_CENSO
NO_REGIAO
CO_REGIAO
NO_UF
SG_UF
CO_UF
NO_MUNICIPIO
CO_MUNICIPIO
NO_REGIAO_GEOG_INTERM
CO_REGIAO_GEOG_INTERM
NO_REGIAO_GEOG_IMED
CO_REGIAO_GEOG_IMED
NO_MESORREGIAO
CO_MESORREGIAO
NO_MICRORREGIAO
CO_MICRORREGIAO
NO_DISTRITO
CO_DISTRITO
NO_ENTIDADE
CO_ENTIDADE
TP_DEPENDENCIA
TP_CATEGORIA_ESCOLA_PRIVADA
TP_LOCALIZACAO
TP_LOCALIZACAO_DIFERENCIADA
DS_ENDERECO
NU_ENDERECO
DS_COMPLEMENTO
NO_BAIRRO
CO_CEP
NU_DDD
NU_TELEFONE
TP_SITUACAO_FUNCIONAMENTO
CO_ORGAO_REGIONAL
DT_ANO_LETIVO_INICIO
DT_ANO_LETIVO_TERMINO
IN_VINCULO_SECRETARIA_EDUCACAO
IN_VINCULO_SEGURANCA_PUBLICA
IN_VINCULO_SECRETARIA_SAUDE
IN_VINCULO_OUTRO_ORGAO
IN_PODER_PUBLICO_PARCERIA
TP_PODER_PUBLICO_PARCERIA
IN_FORMA_CONT_TERMO_COLABORA
IN_FORMA_CONT_TERMO_FOMENTO
IN_FORMA_CONT_ACORDO_COOP
IN_FORMA_CONT_PRESTACAO_SERV
IN_FORMA_CONT_COOP_TEC_FIN
IN_FORMA_CONT_CONSORCIO_PUB
IN_FORMA_CONT_MU_TERMO_COLAB
IN_FORMA_CONT_MU_TERMO_FOMENTO
IN_FORMA_CONT_MU_ACORDO_COOP
IN_FORMA_CONT_MU_PREST_SERV
IN_FORMA_CONT_MU_COOP_TEC_FIN
IN_F

### 6.6 Seleção das variáveis escolares

O Censo Escolar disponibiliza um grande número de atributos relacionados à infraestrutura, recursos pedagógicos, conectividade, matrículas, turmas e profissionais da escola.

Para evitar o aumento excessivo da dimensionalidade, foram selecionadas apenas variáveis com potencial explicativo para o problema de alfabetização.

A seleção prioriza características relacionadas à infraestrutura básica, acesso à tecnologia, recursos pedagógicos, acessibilidade, tamanho da escola e disponibilidade de profissionais.

In [53]:
colunas_escola = [
    "NU_ANO_CENSO",
    "CO_ENTIDADE",
    "TP_DEPENDENCIA",
    "TP_LOCALIZACAO",

    # Infraestrutura básica
    "IN_AGUA_POTAVEL",
    "IN_ENERGIA_REDE_PUBLICA",
    "IN_ESGOTO_REDE_PUBLICA",

    # Recursos educacionais
    "IN_BIBLIOTECA",
    "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_LABORATORIO_CIENCIAS",
    "IN_LABORATORIO_INFORMATICA",
    "IN_QUADRA_ESPORTES",

    # Tecnologia
    "IN_COMPUTADOR",
    "IN_INTERNET",
    "IN_INTERNET_ALUNOS",
    "IN_INTERNET_APRENDIZAGEM",
    "IN_BANDA_LARGA",

    # Acessibilidade
    "IN_BANHEIRO_PNE",
    "IN_ACESSIBILIDADE_RAMPAS",

    # Estrutura física
    "QT_SALAS_UTILIZADAS",
    "QT_SALAS_UTILIZA_CLIMATIZADAS",

    # Matrículas
    "QT_MAT_BAS",
    "QT_MAT_FUND",
    "QT_MAT_FUND_AI",

    # Docentes
    "QT_DOC_BAS",
    "QT_DOC_FUND",
    "QT_DOC_FUND_AI",

    # Turmas
    "QT_TUR_BAS",
    "QT_TUR_FUND",
    "QT_TUR_FUND_AI",
]

In [54]:
colunas_existentes = [
    coluna
    for coluna in colunas_escola
    if coluna in df_censo_2024_amostra.columns
]

colunas_ausentes = [
    coluna
    for coluna in colunas_escola
    if coluna not in df_censo_2024_amostra.columns
]

print("Colunas existentes:", len(colunas_existentes))
print("Colunas ausentes:", len(colunas_ausentes))

print("\nAusentes:")
for coluna in colunas_ausentes:
    print("-", coluna)

Colunas existentes: 30
Colunas ausentes: 0

Ausentes:


### 6.7 Leitura das variáveis selecionadas de 2024

Após a validação do schema, serão carregadas apenas as variáveis previamente selecionadas.

Essa estratégia reduz o consumo de memória e mantém o processamento focado nas características escolares com maior relevância potencial para o problema.

In [55]:
with zipfile.ZipFile(CENSO_2024_PATH, "r") as zip_2024:
    with zip_2024.open(ARQUIVO_CENSO_2024) as arquivo:
        df_censo_2024 = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=colunas_escola,
            low_memory=False
        )

print("Shape Censo Escolar 2024:", df_censo_2024.shape)

Shape Censo Escolar 2024: (215545, 30)


In [56]:
df_censo_2024.head()

,NU_ANO_CENSO,CO_ENTIDADE,TP_DEPENDENCIA,TP_LOCALIZACAO,IN_AGUA_POTAVEL,IN_ENERGIA_REDE_PUBLICA,IN_ESGOTO_REDE_PUBLICA,IN_BANHEIRO_PNE,IN_BIBLIOTECA,IN_BIBLIOTECA_SALA_LEITURA,...,IN_BANDA_LARGA,QT_MAT_BAS,QT_MAT_FUND,QT_MAT_FUND_AI,QT_DOC_BAS,QT_DOC_FUND,QT_DOC_FUND_AI,QT_TUR_BAS,QT_TUR_FUND,QT_TUR_FUND_AI
0,2024,11022558,2,2,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,4.0,4.0,1.0,1.0,1.0,4.0,4.0,4.0
1,2024,11024275,2,1,1.0,1.0,0.0,1.0,1.0,1.0,...,1.0,131.0,0.0,0.0,8.0,0.0,0.0,4.0,0.0,0.0
2,2024,11024291,3,2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024,11024666,3,2,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,169.0,137.0,66.0,13.0,11.0,3.0,10.0,8.0,3.0
4,2024,11024682,2,1,1.0,1.0,0.0,1.0,1.0,1.0,...,1.0,598.0,237.0,0.0,43.0,13.0,0.0,40.0,11.0,0.0


In [57]:
df_censo_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 215545 entries, 0 to 215544
Data columns (total 30 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   NU_ANO_CENSO                   215545 non-null  int64  
 1   CO_ENTIDADE                    215545 non-null  int64  
 2   TP_DEPENDENCIA                 215545 non-null  int64  
 3   TP_LOCALIZACAO                 215545 non-null  int64  
 4   IN_AGUA_POTAVEL                181065 non-null  float64
 5   IN_ENERGIA_REDE_PUBLICA        181065 non-null  float64
 6   IN_ESGOTO_REDE_PUBLICA         181065 non-null  float64
 7   IN_BANHEIRO_PNE                181065 non-null  float64
 8   IN_BIBLIOTECA                  181065 non-null  float64
 9   IN_BIBLIOTECA_SALA_LEITURA     181065 non-null  float64
 10  IN_LABORATORIO_CIENCIAS        181065 non-null  float64
 11  IN_LABORATORIO_INFORMATICA     181065 non-null  float64
 12  IN_QUADRA_ESPORTES             181065 non

In [58]:
print(
    "Escolas únicas:",
    df_censo_2024["CO_ENTIDADE"].nunique()
)

print(
    "Duplicidades por escola:",
    df_censo_2024["CO_ENTIDADE"].duplicated().sum()
)

Escolas únicas: 215545
Duplicidades por escola: 0


In [59]:
ARQUIVO_CENSO_2023 = (
    "microdados_censo_escolar_2023/"
    "dados/microdados_ed_basica_2023.csv"
)

with zipfile.ZipFile(CENSO_2023_PATH, "r") as zip_2023:
    with zip_2023.open(ARQUIVO_CENSO_2023) as arquivo:
        df_censo_2023 = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=colunas_escola,
            low_memory=False
        )

print("Shape Censo Escolar 2023:", df_censo_2023.shape)

print(
    "Escolas únicas:",
    df_censo_2023["CO_ENTIDADE"].nunique()
)

print(
    "Duplicidades por escola:",
    df_censo_2023["CO_ENTIDADE"].duplicated().sum()
)

Shape Censo Escolar 2023: (217625, 30)
Escolas únicas: 217625
Duplicidades por escola: 0


### 6.9 Consolidação dos dados escolares

Após a leitura e validação dos microdados de 2023 e 2024, as duas bases serão consolidadas em um único dataset.

A combinação será realizada preservando o ano de referência e o identificador da escola, permitindo posteriormente a integração com o dataset analítico por `id_escola` e `ano`.

In [60]:
df_escolas = pd.concat(
    [df_censo_2023, df_censo_2024],
    ignore_index=True
)

print("Shape consolidado:", df_escolas.shape)

Shape consolidado: (433170, 30)


In [61]:
df_escolas = df_escolas.rename(
    columns={
        "NU_ANO_CENSO": "ano",
        "CO_ENTIDADE": "id_escola"
    }
)

df_escolas["ano"] = pd.to_numeric(
    df_escolas["ano"],
    errors="coerce"
).astype("Int64")

df_escolas["id_escola"] = (
    df_escolas["id_escola"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

In [62]:
print(df_escolas[["ano", "id_escola"]].dtypes)

print(
    "Duplicidades por escola + ano:",
    df_escolas.duplicated(
        subset=["id_escola", "ano"]
    ).sum()
)

ano           Int64
id_escola    string
dtype: object
Duplicidades por escola + ano: 0


In [63]:
print(
    df_escolas["ano"]
    .value_counts()
    .sort_index()
)

ano
2023    217625
2024    215545
Name: count, dtype: Int64


### 6.10 Padronização dos nomes das variáveis

As variáveis selecionadas do Censo Escolar serão renomeadas para facilitar a interpretação durante a análise exploratória, a engenharia de atributos e a modelagem.

A padronização mantém o significado original das variáveis, mas utiliza nomes mais claros e consistentes com o restante do projeto.

In [64]:
mapa_colunas_escola = {
    "TP_DEPENDENCIA": "dependencia_administrativa",
    "TP_LOCALIZACAO": "localizacao_escola",

    "IN_AGUA_POTAVEL": "tem_agua_potavel",
    "IN_ENERGIA_REDE_PUBLICA": "tem_energia_rede_publica",
    "IN_ESGOTO_REDE_PUBLICA": "tem_esgoto_rede_publica",

    "IN_BIBLIOTECA": "tem_biblioteca",
    "IN_BIBLIOTECA_SALA_LEITURA": "tem_biblioteca_sala_leitura",
    "IN_LABORATORIO_CIENCIAS": "tem_lab_ciencias",
    "IN_LABORATORIO_INFORMATICA": "tem_lab_informatica",
    "IN_QUADRA_ESPORTES": "tem_quadra_esportes",

    "IN_COMPUTADOR": "tem_computador",
    "IN_INTERNET": "tem_internet",
    "IN_INTERNET_ALUNOS": "tem_internet_alunos",
    "IN_INTERNET_APRENDIZAGEM": "tem_internet_aprendizagem",
    "IN_BANDA_LARGA": "tem_banda_larga",

    "IN_BANHEIRO_PNE": "tem_banheiro_pne",
    "IN_ACESSIBILIDADE_RAMPAS": "tem_rampas_acessibilidade",

    "QT_SALAS_UTILIZADAS": "qtd_salas_utilizadas",
    "QT_SALAS_UTILIZA_CLIMATIZADAS": "qtd_salas_climatizadas",

    "QT_MAT_BAS": "qtd_matriculas_basica",
    "QT_MAT_FUND": "qtd_matriculas_fundamental",
    "QT_MAT_FUND_AI": "qtd_matriculas_fundamental_ai",

    "QT_DOC_BAS": "qtd_docentes_basica",
    "QT_DOC_FUND": "qtd_docentes_fundamental",
    "QT_DOC_FUND_AI": "qtd_docentes_fundamental_ai",

    "QT_TUR_BAS": "qtd_turmas_basica",
    "QT_TUR_FUND": "qtd_turmas_fundamental",
    "QT_TUR_FUND_AI": "qtd_turmas_fundamental_ai",
}

df_escolas = df_escolas.rename(
    columns=mapa_colunas_escola
)

In [65]:
print("Quantidade de colunas:", df_escolas.shape[1])

df_escolas.head()

Quantidade de colunas: 30


,ano,id_escola,dependencia_administrativa,localizacao_escola,tem_agua_potavel,tem_energia_rede_publica,tem_esgoto_rede_publica,tem_banheiro_pne,tem_biblioteca,tem_biblioteca_sala_leitura,...,tem_banda_larga,qtd_matriculas_basica,qtd_matriculas_fundamental,qtd_matriculas_fundamental_ai,qtd_docentes_basica,qtd_docentes_fundamental,qtd_docentes_fundamental_ai,qtd_turmas_basica,qtd_turmas_fundamental,qtd_turmas_fundamental_ai
0,2023,11000023,2,1,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,69.0,69.0,69.0,16.0,16.0,16.0,14.0,14.0,14.0
1,2023,11000040,3,1,1.0,1.0,1.0,0.0,0.0,0.0,...,1.0,225.0,0.0,0.0,10.0,0.0,0.0,12.0,0.0,0.0
2,2023,11000058,4,1,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1229.0,835.0,335.0,66.0,42.0,16.0,38.0,27.0,13.0
3,2023,11000082,4,1,1.0,1.0,1.0,0.0,1.0,1.0,...,1.0,44.0,28.0,28.0,6.0,4.0,4.0,5.0,3.0,3.0
4,2023,11000104,4,1,1.0,1.0,0.0,1.0,1.0,1.0,...,1.0,640.0,538.0,295.0,23.0,20.0,11.0,27.0,22.0,13.0


In [66]:
df_escolas.columns.tolist()

['ano',
 'id_escola',
 'dependencia_administrativa',
 'localizacao_escola',
 'tem_agua_potavel',
 'tem_energia_rede_publica',
 'tem_esgoto_rede_publica',
 'tem_banheiro_pne',
 'tem_biblioteca',
 'tem_biblioteca_sala_leitura',
 'tem_lab_ciencias',
 'tem_lab_informatica',
 'tem_quadra_esportes',
 'tem_rampas_acessibilidade',
 'qtd_salas_utilizadas',
 'qtd_salas_climatizadas',
 'tem_computador',
 'tem_internet',
 'tem_internet_alunos',
 'tem_internet_aprendizagem',
 'tem_banda_larga',
 'qtd_matriculas_basica',
 'qtd_matriculas_fundamental',
 'qtd_matriculas_fundamental_ai',
 'qtd_docentes_basica',
 'qtd_docentes_fundamental',
 'qtd_docentes_fundamental_ai',
 'qtd_turmas_basica',
 'qtd_turmas_fundamental',
 'qtd_turmas_fundamental_ai']

### 6.11 Tradução das variáveis categóricas

As variáveis de dependência administrativa e localização da escola são originalmente representadas por códigos numéricos no Censo Escolar.

Para facilitar a interpretação durante a análise exploratória e a modelagem, esses códigos serão convertidos em categorias textuais.

In [67]:
mapa_dependencia = {
    1: "Federal",
    2: "Estadual",
    3: "Municipal",
    4: "Privada"
}

mapa_localizacao = {
    1: "Urbana",
    2: "Rural"
}

df_escolas["dependencia_administrativa"] = (
    df_escolas["dependencia_administrativa"]
    .map(mapa_dependencia)
    .astype("string")
)

df_escolas["localizacao_escola"] = (
    df_escolas["localizacao_escola"]
    .map(mapa_localizacao)
    .astype("string")
)

In [68]:
print(
    df_escolas["dependencia_administrativa"]
    .value_counts(dropna=False)
)

print()

print(
    df_escolas["localizacao_escola"]
    .value_counts(dropna=False)
)

dependencia_administrativa
Municipal    259708
Privada      105213
Estadual      66800
Federal        1449
Name: count, dtype: int64[pyarrow]

localizacao_escola
Urbana    288618
Rural     144552
Name: count, dtype: int64[pyarrow]


### 6.12 Criação de indicadores escolares derivados

Além das variáveis originais do Censo Escolar, serão criados indicadores derivados para representar melhor a estrutura e a disponibilidade de recursos das escolas.

Esses indicadores transformam contagens absolutas em métricas relativas, facilitando comparações entre escolas de diferentes portes.

In [69]:
import numpy as np

df_escolas["alunos_por_turma_fund_ai"] = np.where(
    df_escolas["qtd_turmas_fundamental_ai"] > 0,
    df_escolas["qtd_matriculas_fundamental_ai"]
    / df_escolas["qtd_turmas_fundamental_ai"],
    np.nan
)

df_escolas["alunos_por_docente_fund_ai"] = np.where(
    df_escolas["qtd_docentes_fundamental_ai"] > 0,
    df_escolas["qtd_matriculas_fundamental_ai"]
    / df_escolas["qtd_docentes_fundamental_ai"],
    np.nan
)

df_escolas["proporcao_salas_climatizadas"] = np.where(
    df_escolas["qtd_salas_utilizadas"] > 0,
    df_escolas["qtd_salas_climatizadas"]
    / df_escolas["qtd_salas_utilizadas"],
    np.nan
)

In [70]:
df_escolas[
    [
        "alunos_por_turma_fund_ai",
        "alunos_por_docente_fund_ai",
        "proporcao_salas_climatizadas"
    ]
].describe()

,alunos_por_turma_fund_ai,alunos_por_docente_fund_ai,proporcao_salas_climatizadas
count,178962.000000,178958.000000,361295.000000
mean,19.350947,16.072541,0.352530
std,9.621388,9.111923,0.450401
min,0.000000,0.000000,0.000000
25%,13.200000,10.000000,0.000000
50%,19.300000,14.900000,0.000000
75%,24.000000,20.300000,1.000000
max,337.000000,252.000000,4.500000


In [71]:
print(
    df_escolas[
        [
            "alunos_por_turma_fund_ai",
            "alunos_por_docente_fund_ai",
            "proporcao_salas_climatizadas"
        ]
    ].isna().mean().mul(100).round(2)
)

alunos_por_turma_fund_ai        58.69
alunos_por_docente_fund_ai      58.69
proporcao_salas_climatizadas    16.59
dtype: float64


### 6.13 Validação dos indicadores derivados

Antes da integração com a base analítica, os indicadores derivados serão avaliados quanto à presença de valores extremos ou inconsistências.

Essa validação é especialmente importante para métricas calculadas a partir de razões entre variáveis, pois diferenças de preenchimento ou características específicas das escolas podem gerar valores fora dos intervalos esperados.

In [72]:
print(
    "Proporção de salas climatizadas > 1:",
    (df_escolas["proporcao_salas_climatizadas"] > 1).sum()
)

print(
    "Alunos por turma > 100:",
    (df_escolas["alunos_por_turma_fund_ai"] > 100).sum()
)

print(
    "Alunos por docente > 100:",
    (df_escolas["alunos_por_docente_fund_ai"] > 100).sum()
)

Proporção de salas climatizadas > 1: 1
Alunos por turma > 100: 61
Alunos por docente > 100: 23


In [73]:
df_escolas.loc[
    df_escolas["proporcao_salas_climatizadas"] > 1,
    [
        "ano",
        "id_escola",
        "qtd_salas_utilizadas",
        "qtd_salas_climatizadas",
        "proporcao_salas_climatizadas"
    ]
].sort_values(
    "proporcao_salas_climatizadas",
    ascending=False
).head(20)

,ano,id_escola,qtd_salas_utilizadas,qtd_salas_climatizadas,proporcao_salas_climatizadas
346084,2024,33141118,2.0,9.0,4.5


### 6.14 Validação da cobertura das escolas

Antes da integração dos dados escolares, será verificada a correspondência entre as escolas presentes no dataset analítico e os registros do Censo Escolar.

Essa etapa permite avaliar a cobertura do enriquecimento e verificar se eventuais valores extremos identificados nos microdados estão presentes no universo efetivamente utilizado no projeto.

In [74]:
df_enriquecido["ano"] = pd.to_numeric(
    df_enriquecido["ano"],
    errors="coerce"
).astype("Int64")

df_enriquecido["id_escola"] = (
    df_enriquecido["id_escola"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

In [75]:
print(
    df_enriquecido[
        ["ano", "id_escola"]
    ].dtypes
)

print(
    df_escolas[
        ["ano", "id_escola"]
    ].dtypes
)

ano           Int64
id_escola    string
dtype: object
ano           Int64
id_escola    string
dtype: object


In [76]:
chaves_escolas = df_escolas[
    ["ano", "id_escola"]
].drop_duplicates()

df_cobertura_escolas = df_enriquecido[
    ["ano", "id_escola"]
].merge(
    chaves_escolas.assign(encontrou_censo=1),
    on=["ano", "id_escola"],
    how="left",
    validate="many_to_one"
)

print(
    "Cobertura geral:",
    round(
        df_cobertura_escolas["encontrou_censo"]
        .notna()
        .mean()
        * 100,
        2
    ),
    "%"
)

Cobertura geral: 0.0 %


In [77]:
print(
    df_cobertura_escolas
    .assign(
        encontrou_censo=lambda x:
            x["encontrou_censo"].notna()
    )
    .groupby("ano")["encontrou_censo"]
    .agg(["count", "sum", "mean"])
)

      count  sum  mean
ano                   
2023  28295    0   0.0
2024  29487    0   0.0


In [78]:
escolas_projeto = set(
    zip(
        df_enriquecido["ano"],
        df_enriquecido["id_escola"]
    )
)

df_extremos_projeto = df_escolas[
    df_escolas.apply(
        lambda linha: (
            linha["ano"],
            linha["id_escola"]
        ) in escolas_projeto,
        axis=1
    )
].copy()

print(
    "Salas climatizadas > 1:",
    (
        df_extremos_projeto[
            "proporcao_salas_climatizadas"
        ] > 1
    ).sum()
)

print(
    "Alunos por turma > 100:",
    (
        df_extremos_projeto[
            "alunos_por_turma_fund_ai"
        ] > 100
    ).sum()
)

print(
    "Alunos por docente > 100:",
    (
        df_extremos_projeto[
            "alunos_por_docente_fund_ai"
        ] > 100
    ).sum()
)

Salas climatizadas > 1: 0
Alunos por turma > 100: 0
Alunos por docente > 100: 0


In [79]:
print("Amostra id_escola - projeto:")
print(df_enriquecido["id_escola"].head(10).tolist())

print("\nAmostra id_escola - Censo:")
print(df_escolas["id_escola"].head(10).tolist())

print("\nTamanho dos códigos - projeto:")
print(
    df_enriquecido["id_escola"]
    .str.len()
    .value_counts()
    .sort_index()
)

print("\nTamanho dos códigos - Censo:")
print(
    df_escolas["id_escola"]
    .str.len()
    .value_counts()
    .sort_index()
)

Amostra id_escola - projeto:
['60000951', '60000963', '60001351', '60004115', '60004434', '60011396', '60014022', '60017147', '60017630', '60018113']

Amostra id_escola - Censo:
['11000023', '11000040', '11000058', '11000082', '11000104', '11000171', '11000180', '11000198', '11000201', '11000252']

Tamanho dos códigos - projeto:
id_escola
8    57782
Name: count, dtype: Int64

Tamanho dos códigos - Censo:
id_escola
8    433170
Name: count, dtype: Int64


In [80]:
print(
    "Exemplo projeto:",
    df_enriquecido["id_escola"].dropna().iloc[0]
)

print(
    "Exemplo Censo:",
    df_escolas["id_escola"].dropna().iloc[0]
)

Exemplo projeto: 60000951
Exemplo Censo: 11000023


### 6.15 Limitação da integração por escola

A tentativa de integração direta entre o dataset analítico e o Censo Escolar por identificador da escola apresentou cobertura de 0%.

A análise dos identificadores indicou que `id_escola`, presente na base de alfabetização, não corresponde ao código oficial `CO_ENTIDADE` utilizado pelo INEP.

Dessa forma, para evitar associações incorretas, optou-se por utilizar os dados do Censo Escolar de forma agregada por município, ano e dependência administrativa.

Essa estratégia preserva informações relevantes sobre o contexto educacional local sem assumir uma correspondência inexistente entre identificadores de escola.

In [81]:
colunas_contexto_escolar = [
    "NU_ANO_CENSO",
    "CO_MUNICIPIO",
    "TP_DEPENDENCIA",
    "TP_LOCALIZACAO",

    "IN_AGUA_POTAVEL",
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_LABORATORIO_INFORMATICA",
    "IN_INTERNET",
    "IN_INTERNET_APRENDIZAGEM",
    "IN_BANDA_LARGA",
    "IN_ACESSIBILIDADE_RAMPAS",

    "QT_SALAS_UTILIZADAS",
    "QT_SALAS_UTILIZA_CLIMATIZADAS",

    "QT_MAT_FUND_AI",
    "QT_DOC_FUND_AI",
    "QT_TUR_FUND_AI",
]

In [82]:
with zipfile.ZipFile(CENSO_2023_PATH, "r") as zip_2023:
    with zip_2023.open(ARQUIVO_CENSO_2023) as arquivo:
        df_contexto_2023 = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=colunas_contexto_escolar,
            low_memory=False
        )

with zipfile.ZipFile(CENSO_2024_PATH, "r") as zip_2024:
    with zip_2024.open(ARQUIVO_CENSO_2024) as arquivo:
        df_contexto_2024 = pd.read_csv(
            arquivo,
            sep=";",
            encoding="latin1",
            usecols=colunas_contexto_escolar,
            low_memory=False
        )

df_contexto_escolar = pd.concat(
    [df_contexto_2023, df_contexto_2024],
    ignore_index=True
)

print("Shape:", df_contexto_escolar.shape)

Shape: (433170, 17)


In [83]:
df_contexto_escolar = df_contexto_escolar.rename(
    columns={
        "NU_ANO_CENSO": "ano",
        "CO_MUNICIPIO": "id_municipio"
    }
)

df_contexto_escolar["ano"] = pd.to_numeric(
    df_contexto_escolar["ano"],
    errors="coerce"
).astype("Int64")

df_contexto_escolar["id_municipio"] = (
    df_contexto_escolar["id_municipio"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(7)
)

mapa_rede = {
    1: "Federal",
    2: "Estadual",
    3: "Municipal",
    4: "Privada"
}

df_contexto_escolar["rede"] = (
    df_contexto_escolar["TP_DEPENDENCIA"]
    .map(mapa_rede)
    .astype("string")
)

In [84]:
df_contexto_escolar = df_contexto_escolar[
    df_contexto_escolar["rede"].isin(
        ["Municipal", "Estadual"]
    )
].copy()

In [85]:
df_contexto_escolar["escola_rural"] = (
    df_contexto_escolar["TP_LOCALIZACAO"] == 2
).astype(int)

df_contexto_escolar["alunos_por_turma_fund_ai"] = np.where(
    df_contexto_escolar["QT_TUR_FUND_AI"] > 0,
    df_contexto_escolar["QT_MAT_FUND_AI"]
    / df_contexto_escolar["QT_TUR_FUND_AI"],
    np.nan
)

df_contexto_escolar["alunos_por_docente_fund_ai"] = np.where(
    df_contexto_escolar["QT_DOC_FUND_AI"] > 0,
    df_contexto_escolar["QT_MAT_FUND_AI"]
    / df_contexto_escolar["QT_DOC_FUND_AI"],
    np.nan
)

df_contexto_escolar["proporcao_salas_climatizadas"] = np.where(
    df_contexto_escolar["QT_SALAS_UTILIZADAS"] > 0,
    df_contexto_escolar["QT_SALAS_UTILIZA_CLIMATIZADAS"]
    / df_contexto_escolar["QT_SALAS_UTILIZADAS"],
    np.nan
)

# Razões acima de 1 são consideradas inconsistentes
df_contexto_escolar.loc[
    df_contexto_escolar["proporcao_salas_climatizadas"] > 1,
    "proporcao_salas_climatizadas"
] = np.nan

In [86]:
df_contexto_municipal_escolar = (
    df_contexto_escolar
    .groupby(
        ["id_municipio", "ano", "rede"],
        as_index=False
    )
    .agg(
        qtd_escolas_censo=("rede", "size"),

        prop_escolas_rurais=(
            "escola_rural", "mean"
        ),

        prop_escolas_agua_potavel=(
            "IN_AGUA_POTAVEL", "mean"
        ),

        prop_escolas_esgoto_rede=(
            "IN_ESGOTO_REDE_PUBLICA", "mean"
        ),

        prop_escolas_biblioteca_leitura=(
            "IN_BIBLIOTECA_SALA_LEITURA", "mean"
        ),

        prop_escolas_lab_informatica=(
            "IN_LABORATORIO_INFORMATICA", "mean"
        ),

        prop_escolas_internet=(
            "IN_INTERNET", "mean"
        ),

        prop_escolas_internet_aprendizagem=(
            "IN_INTERNET_APRENDIZAGEM", "mean"
        ),

        prop_escolas_banda_larga=(
            "IN_BANDA_LARGA", "mean"
        ),

        prop_escolas_rampas_acessibilidade=(
            "IN_ACESSIBILIDADE_RAMPAS", "mean"
        ),

        media_alunos_turma_fund_ai=(
            "alunos_por_turma_fund_ai", "mean"
        ),

        media_alunos_docente_fund_ai=(
            "alunos_por_docente_fund_ai", "mean"
        ),

        media_prop_salas_climatizadas=(
            "proporcao_salas_climatizadas", "mean"
        )
    )
)

print(
    "Shape contexto escolar agregado:",
    df_contexto_municipal_escolar.shape
)

print(
    "Duplicidades na chave:",
    df_contexto_municipal_escolar.duplicated(
        ["id_municipio", "ano", "rede"]
    ).sum()
)

Shape contexto escolar agregado: (22264, 16)
Duplicidades na chave: 0


In [87]:
shape_antes = df_enriquecido.shape

df_enriquecido = df_enriquecido.merge(
    df_contexto_municipal_escolar,
    on=["id_municipio", "ano", "rede"],
    how="left",
    validate="many_to_one"
)

shape_depois = df_enriquecido.shape

print("Shape antes:", shape_antes)
print("Shape depois:", shape_depois)
print(
    "Diferença de linhas:",
    shape_depois[0] - shape_antes[0]
)

Shape antes: (57782, 22)
Shape depois: (57782, 35)
Diferença de linhas: 0


In [88]:
colunas_contexto = [
    "qtd_escolas_censo",
    "prop_escolas_rurais",
    "prop_escolas_internet",
    "prop_escolas_banda_larga",
    "media_alunos_turma_fund_ai",
    "media_alunos_docente_fund_ai"
]

for coluna in colunas_contexto:
    cobertura = (
        df_enriquecido[coluna]
        .notna()
        .mean()
        * 100
    )

    print(
        f"{coluna}: {cobertura:.2f}%"
    )

qtd_escolas_censo: 100.00%
prop_escolas_rurais: 100.00%
prop_escolas_internet: 100.00%
prop_escolas_banda_larga: 100.00%
media_alunos_turma_fund_ai: 99.99%
media_alunos_docente_fund_ai: 99.99%


In [89]:
OUTPUT_ENRIQUECIDO = (
    GOLD_DIR / "gold_dataset_enriquecido.parquet"
)

df_enriquecido.to_parquet(
    OUTPUT_ENRIQUECIDO,
    index=False
)

print("Dataset salvo em:", OUTPUT_ENRIQUECIDO)
print("Shape final:", df_enriquecido.shape)

Dataset salvo em: ..\data\gold\gold_dataset_enriquecido.parquet
Shape final: (57782, 35)


In [90]:
df_enriquecido.head()

,ano,id_municipio,id_municipio_nome,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,...,prop_escolas_esgoto_rede,prop_escolas_biblioteca_leitura,prop_escolas_lab_informatica,prop_escolas_internet,prop_escolas_internet_aprendizagem,prop_escolas_banda_larga,prop_escolas_rampas_acessibilidade,media_alunos_turma_fund_ai,media_alunos_docente_fund_ai,media_prop_salas_climatizadas
0,2023,1302504,Manacapuru,60000951,13015851,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,...,0.007353,0.132353,0.029412,0.705882,0.411765,0.541667,0.308824,19.188693,16.360779,0.642714
1,2023,1302603,Manaus,60000963,13030738,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,...,0.258517,0.531062,0.529058,0.969940,0.665331,0.878099,0.492986,25.931574,28.238906,0.920097
2,2023,1300631,Beruri,60001351,13003982,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,...,0.000000,0.044944,0.000000,0.292135,0.224719,0.884615,0.056180,15.716064,14.607079,0.088443
3,2023,1711506,Jaú do Tocantins,60004115,17012510,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,...,0.000000,0.200000,0.200000,1.000000,0.200000,0.400000,0.600000,17.472222,17.472222,0.200000
4,2023,2100709,Anajatuba,60004434,21012344,1,2° ano do Ensino Fundamental,Municipal,Ausente,Prova não preenchida,...,0.051282,0.153846,0.000000,0.948718,0.179487,1.000000,0.384615,25.611508,16.573413,0.198077


## 7. Enriquecimento socioeconômico municipal

Nesta etapa, o dataset será enriquecido com indicadores socioeconômicos em nível municipal.

O objetivo é complementar as variáveis territoriais e econômicas já existentes com informações mais diretamente relacionadas às condições de vida da população, buscando ampliar o poder explicativo das features utilizadas na modelagem.

A integração será realizada por `id_municipio`, preservando integralmente os registros da base principal.

In [91]:
SOCIO_DIR = EXTERNAL_DIR / "socioeconomico"
SOCIO_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório socioeconômico:", SOCIO_DIR)

Diretório socioeconômico: ..\data\external\socioeconomico


In [92]:
print(
    df_enriquecido["id_municipio"]
    .astype("string")
    .str.len()
    .value_counts()
    .sort_index()
)

print(
    "Municípios únicos:",
    df_enriquecido["id_municipio"].nunique()
)

id_municipio
7    57782
Name: count, dtype: Int64
Municípios únicos: 4591


### 7.1 Rendimento domiciliar per capita

O PIB per capita representa a produção econômica do município, mas não necessariamente a renda efetivamente disponível às famílias.

Para complementar essa dimensão, será incorporado um indicador de rendimento domiciliar per capita do Censo Demográfico 2022, permitindo representar de forma mais direta as condições econômicas da população residente.

In [93]:
import requests
import pandas as pd

URL_RENDA = (
    "https://apisidra.ibge.gov.br/values/"
    "t/10295/n6/all/v/all/p/2022"
)

response_renda = requests.get(
    URL_RENDA,
    timeout=60
)

print("Status:", response_renda.status_code)

Status: 200


In [94]:
dados_renda = response_renda.json()

df_renda_raw = pd.DataFrame(
    dados_renda
)

print("Shape:", df_renda_raw.shape)

df_renda_raw.head()

Shape: (22281, 17)


,NC,NN,MC,MN,V,D1C,D1N,D2C,D2N,D3C,D3N,D4C,D4N,D5C,D5N,D6C,D6N
0,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Sexo (Código),Sexo,Grupo de idade (Código),Grupo de idade,Cor ou raça (Código),Cor ou raça
1,6,Município,45,Pessoas,21447,1100015,Alta Floresta D'Oeste - RO,13604,Moradores em domicílios particulares permanent...,2022,2022,6794,Total,95253,Total,95251,Total
2,6,Município,2,Percentual,100.00,1100015,Alta Floresta D'Oeste - RO,1013604,Moradores em domicílios particulares permanent...,2022,2022,6794,Total,95253,Total,95251,Total
3,6,Município,38,Reais,1210.60,1100015,Alta Floresta D'Oeste - RO,13431,Valor do rendimento nominal médio mensal domic...,2022,2022,6794,Total,95253,Total,95251,Total
4,6,Município,38,Reais,920.00,1100015,Alta Floresta D'Oeste - RO,13534,Valor do rendimento nominal mediano mensal dom...,2022,2022,6794,Total,95253,Total,95251,Total


In [95]:
df_renda_raw.columns.tolist()

['NC',
 'NN',
 'MC',
 'MN',
 'V',
 'D1C',
 'D1N',
 'D2C',
 'D2N',
 'D3C',
 'D3N',
 'D4C',
 'D4N',
 'D5C',
 'D5N',
 'D6C',
 'D6N']

In [96]:
variaveis_renda = (
    df_renda_raw[
        ["D2C", "D2N"]
    ]
    .drop_duplicates()
    .sort_values("D2C")
)

print("Quantidade de variáveis:", len(variaveis_renda))

display(variaveis_renda)

Quantidade de variáveis: 5


,D2C,D2N
2,1013604,Moradores em domicílios particulares permanent...
3,13431,Valor do rendimento nominal médio mensal domic...
4,13534,Valor do rendimento nominal mediano mensal dom...
1,13604,Moradores em domicílios particulares permanent...
0,Variável (Código),Variável


In [97]:
filtro_renda = variaveis_renda[
    variaveis_renda["D2N"]
    .astype("string")
    .str.contains(
        "rendimento|renda",
        case=False,
        na=False
    )
]

display(filtro_renda)

,D2C,D2N
3,13431,Valor do rendimento nominal médio mensal domic...
4,13534,Valor do rendimento nominal mediano mensal dom...


In [98]:
pd.set_option("display.max_colwidth", None)

display(
    filtro_renda[
        ["D2C", "D2N"]
    ]
)

,D2C,D2N
3,13431,"Valor do rendimento nominal médio mensal domiciliar per capita dos moradores em domicílios particulares permanentes ocupados, exclusive os cuja condição no domicílio era pensionista, empregado(a) doméstico(a) ou parente do(a) empregado(a) doméstico(a)"
4,13534,"Valor do rendimento nominal mediano mensal domiciliar per capita dos moradores em domicílios particulares permanentes ocupados, exclusive os cuja condição no domicílio era pensionista, empregado(a) doméstico(a) ou parente do(a) empregado(a) doméstico(a)"


### 7.2 Seleção do indicador de renda

Para representar a condição socioeconômica dos municípios, foi selecionado o valor do rendimento nominal mediano mensal domiciliar per capita.

A mediana foi priorizada em relação à média por apresentar maior robustez a valores extremos e à concentração de renda, representando melhor o nível de rendimento típico da população residente no município.

Foram considerados apenas os registros referentes ao total da população, sem segmentação por sexo, grupo de idade ou cor/raça.

In [99]:
df_renda_mediana = df_renda_raw[
    (df_renda_raw["D2C"].astype(str) == "13534")
    & (df_renda_raw["D4N"] == "Total")
    & (df_renda_raw["D5N"] == "Total")
    & (df_renda_raw["D6N"] == "Total")
].copy()

print("Shape:", df_renda_mediana.shape)

display(
    df_renda_mediana[
        [
            "D1C",
            "D1N",
            "MC",
            "MN",
            "V"
        ]
    ].head(10)
)

Shape: (5570, 17)


,D1C,D1N,MC,MN,V
4,1100015,Alta Floresta D'Oeste - RO,38,Reais,920.00
8,1100023,Ariquemes - RO,38,Reais,1066.67
12,1100031,Cabixi - RO,38,Reais,1033.33
16,1100049,Cacoal - RO,38,Reais,1200.00
20,1100056,Cerejeiras - RO,38,Reais,1200.00
24,1100064,Colorado do Oeste - RO,38,Reais,1200.00
28,1100072,Corumbiara - RO,38,Reais,981.00
32,1100080,Costa Marques - RO,38,Reais,775.00
36,1100098,Espigão D'Oeste - RO,38,Reais,1080.00
40,1100106,Guajará-Mirim - RO,38,Reais,583.33


In [100]:
print(
    "Municípios únicos:",
    df_renda_mediana["D1C"].nunique()
)

print(
    "Duplicidades por município:",
    df_renda_mediana["D1C"].duplicated().sum()
)

print("\nUnidades de medida:")
print(
    df_renda_mediana[
        ["MC", "MN"]
    ].drop_duplicates()
)

Municípios únicos: 5570
Duplicidades por município: 0

Unidades de medida:
   MC     MN
4  38  Reais


In [101]:
df_renda_contexto = (
    df_renda_mediana[
        ["D1C", "V"]
    ]
    .rename(
        columns={
            "D1C": "id_municipio",
            "V": "renda_domiciliar_per_capita_mediana"
        }
    )
    .copy()
)

df_renda_contexto["id_municipio"] = (
    df_renda_contexto["id_municipio"]
    .astype("string")
    .str.zfill(7)
)

df_renda_contexto["renda_domiciliar_per_capita_mediana"] = (
    pd.to_numeric(
        df_renda_contexto["renda_domiciliar_per_capita_mediana"],
        errors="coerce"
    )
)

df_renda_contexto.head()

,id_municipio,renda_domiciliar_per_capita_mediana
4,1100015,920.00
8,1100023,1066.67
12,1100031,1033.33
16,1100049,1200.00
20,1100056,1200.00


In [102]:
print("Shape:", df_renda_contexto.shape)

print(
    "Nulos:",
    df_renda_contexto[
        "renda_domiciliar_per_capita_mediana"
    ].isna().sum()
)

print(
    "Duplicidades:",
    df_renda_contexto["id_municipio"]
    .duplicated()
    .sum()
)

Shape: (5570, 2)
Nulos: 0
Duplicidades: 0


In [103]:
shape_antes = df_enriquecido.shape

df_enriquecido = df_enriquecido.merge(
    df_renda_contexto,
    on="id_municipio",
    how="left",
    validate="many_to_one"
)

shape_depois = df_enriquecido.shape

print("Shape antes:", shape_antes)
print("Shape depois:", shape_depois)
print(
    "Diferença de linhas:",
    shape_depois[0] - shape_antes[0]
)

Shape antes: (57782, 35)
Shape depois: (57782, 36)
Diferença de linhas: 0


In [104]:
cobertura_renda = (
    df_enriquecido[
        "renda_domiciliar_per_capita_mediana"
    ]
    .notna()
    .mean()
    * 100
)

print(
    "Cobertura renda:",
    round(cobertura_renda, 2),
    "%"
)

Cobertura renda: 100.0 %


### 7.5 Avaliação de indicadores adicionais de vulnerabilidade econômica

Foi avaliada a possibilidade de incorporar um indicador municipal de baixa renda baseado em classes de rendimento domiciliar per capita.

Entretanto, as tabelas disponíveis com faixas de rendimento não apresentaram, de forma direta e compatível, o mesmo recorte municipal e temporal necessário para integração segura com o Censo Demográfico 2022.

Para preservar a consistência metodológica do projeto, optou-se por não incorporar indicadores cuja granularidade ou período não fossem plenamente compatíveis com a base principal.

### 7.6 Salvamento do dataset enriquecido atualizado

Após a incorporação do indicador de rendimento domiciliar per capita mediano, o dataset enriquecido é salvo novamente na camada Gold.

A atualização preserva os registros existentes e adiciona a nova dimensão socioeconômica, permitindo que as etapas posteriores de análise exploratória e modelagem utilizem a versão mais completa da base.

In [112]:
OUTPUT_ENRIQUECIDO = (
    GOLD_DIR / "gold_dataset_enriquecido.parquet"
)

df_enriquecido.to_parquet(
    OUTPUT_ENRIQUECIDO,
    index=False
)

print("Dataset atualizado salvo em:", OUTPUT_ENRIQUECIDO)
print("Shape final:", df_enriquecido.shape)

Dataset atualizado salvo em: ..\data\gold\gold_dataset_enriquecido.parquet
Shape final: (57782, 36)


## 8. Conclusão do enriquecimento dos dados

A etapa de enriquecimento foi concluída com sucesso, ampliando o dataset analítico com variáveis econômicas, socioeconômicas, populacionais, territoriais e educacionais.

Foram incorporadas informações como PIB per capita, rendimento domiciliar per capita mediano, população, área territorial, densidade demográfica e indicadores agregados do Censo Escolar relacionados à infraestrutura, conectividade e organização das escolas.

A integração preservou integralmente os **57.782 registros** da base original, sem criação de duplicidades e com elevada cobertura para as novas variáveis adicionadas.

Com isso, o dataset passa a representar de forma mais completa diferentes dimensões potencialmente associadas à alfabetização, incluindo contexto econômico, condições socioeconômicas, características territoriais e estrutura educacional.

A base enriquecida encontra-se, portanto, adequada para as etapas seguintes de Análise Exploratória de Dados e modelagem supervisionada.